In [2]:
import tensorflow as tf # tensorflow framework
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPool2D
from keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import splitfolders

splitfolders.ratio(r"C:\Users\Luis\anaconda3\AB\Training", output=r"C:\Users\Luis\anaconda3\outputAB",
    seed=1337, ratio=(.7, .3), group_prefix=None, move=False)



train_path =r"C:\Users\Luis\anaconda3\outputAB\train"
validation_path =r"C:\Users\Luis\anaconda3\outputAB\val"
test_path = r"C:\Users\Luis\anaconda3\AB\Test"


train_datagen = ImageDataGenerator(rescale=1./255,
                   shear_range=0.3,
                   horizontal_flip=True,
                   zoom_range=0.3)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)
batch_size= 32

train_Generator = train_datagen.flow_from_directory(train_path,
                                                          target_size=[100,100],
                                                          batch_size= batch_size,
                                                          color_mode="rgb",
                                                          class_mode="binary",
                                                          shuffle=True)
val_Generator = val_datagen.flow_from_directory(validation_path,
                                                  target_size=[100,100],
                                                    batch_size=batch_size,
                                                    color_mode="rgb",
                                                    class_mode="binary",
                                                    shuffle=False)
test_Generator = test_datagen.flow_from_directory(test_path,
                                                  target_size=[100,100],
                                                    batch_size=batch_size,
                                                    color_mode="rgb",
                                                    class_mode="binary")

model = Sequential()
model.add(Conv2D(16,kernel_size=(5,5),input_shape=(100,100,3),activation="relu"))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Flatten())
model.add(Dense(1,activation="sigmoid"))

model.compile(loss="binary_crossentropy",optimizer="adam",metrics=["accuracy"])

history = model.fit(train_Generator, epochs=10, validation_data=val_Generator)

Copying files: 982 files [00:00, 1049.51 files/s]


Found 687 images belonging to 2 classes.
Found 295 images belonging to 2 classes.
Found 330 images belonging to 2 classes.
Epoch 1/10
22/22 [==============================] - 3s 106ms/step - loss: 0.1130 - accuracy: 0.9534 - val_loss: 9.6314e-06 - val_accuracy: 1.0000
Epoch 2/10
22/22 [==============================] - 2s 99ms/step - loss: 0.0037 - accuracy: 0.9985 - val_loss: 7.8175e-07 - val_accuracy: 1.0000
Epoch 3/10
22/22 [==============================] - 2s 98ms/step - loss: 0.0181 - accuracy: 0.9956 - val_loss: 1.8338e-05 - val_accuracy: 1.0000
Epoch 4/10
22/22 [==============================] - 2s 97ms/step - loss: 0.0576 - accuracy: 0.9825 - val_loss: 2.1484e-08 - val_accuracy: 1.0000
Epoch 5/10
22/22 [==============================] - 2s 96ms/step - loss: 0.0033 - accuracy: 0.9985 - val_loss: 3.1970e-05 - val_accuracy: 1.0000
Epoch 6/10
22/22 [==============================] - 2s 97ms/step - loss: 0.0549 - accuracy: 0.9840 - val_loss: 6.3280e-04 - val_accuracy: 1.0000
Epoch 

In [3]:
model.evaluate(test_Generator)

11/11 [==============================] - 0s 28ms/step - loss: 2.8655e-07 - accuracy: 1.0000


[2.8654986294895934e-07, 1.0]

In [4]:
model.save(r"C:\Users\Luis\anaconda3\savedmodels\ABCategorical.keras")

In [5]:
# Convert the model.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model.
with open('AppleBanana_model.tflite', 'wb') as f:
  f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\Luis\AppData\Local\Temp\tmpb7atnrv_\assets


In [8]:
from tflite_support.metadata_writers import image_classifier
from tflite_support.metadata_writers import writer_utils

In [9]:
ImageClassifierWriter = image_classifier.MetadataWriter
_MODEL_PATH = r"C:\Users\Luis\anaconda3\AppleBanana_model.tflite"
# Task Library expects label files that are in the same format as the one below.
_LABEL_FILE = r"C:\Users\Luis\anaconda3\labelfile.txt"
_SAVE_TO_PATH = r"C:\Users\Luis\anaconda3\categoricaltestmodel.tflite"
# Normalization parameters is required when reprocessing the image. It is
# optional if the image pixel values are in range of [0, 255] and the input
# tensor is quantized to uint8. See the introduction for normalization and
# quantization parameters below for more details.
# https://www.tensorflow.org/lite/models/convert/metadata#normalization_and_quantization_parameters)
_INPUT_NORM_MEAN = 0.5
_INPUT_NORM_STD = 0.5

# Create the metadata writer.
writer = ImageClassifierWriter.create_for_inference(
    writer_utils.load_file(_MODEL_PATH), [_INPUT_NORM_MEAN], [_INPUT_NORM_STD],
    [_LABEL_FILE])

# Verify the metadata generated by metadata writer.
print(writer.get_metadata_json())

# Populate the metadata into the model.
writer_utils.save_file(writer.populate(), _SAVE_TO_PATH)

{
  "name": "ImageClassifier",
  "description": "Identify the most prominent object in the image from a known set of categories.",
  "subgraph_metadata": [
    {
      "input_tensor_metadata": [
        {
          "name": "image",
          "description": "Input image to be classified.",
          "content": {
            "content_properties_type": "ImageProperties",
            "content_properties": {
              "color_space": "RGB"
            }
          },
          "process_units": [
            {
              "options_type": "NormalizationOptions",
              "options": {
                "mean": [
                  127.5
                ],
                "std": [
                  127.5
                ]
              }
            }
          ],
          "stats": {
            "max": [
              1.0
            ],
            "min": [
              -1.0
            ]
          }
        }
      ],
      "output_tensor_metadata": [
        {
          "name": "proba